# 03 – Cost Simulation

> **Research prototype notebook — directional only.**  This notebook documents
> a simple illustrative cost model for the THUNBIT detector.  All cost
> assumptions are hypothetical.  The results should not be interpreted as
> production-value estimates.  They are provided to show *how to think about
> the economic framing*, not to quantify actual savings.

## What this notebook covers

1. Cost assumptions and their justification
2. Two cost components: false-alert review cost and missed-detection stockout cost
3. Illustrative outcome calculation across scenarios
4. Sensitivity to assumptions
5. Clear caveats about what is and is not shown

## Setup

```bash
pip install -e .   # from the repository root
```

In [ ]:
import numpy as np
import pandas as pd

from thunbit import DemandStateDetector, StabilizedDemandDetectorV43

print('Setup complete.')

## 1. Cost assumptions

We model two cost types:

### False-alert review cost
Each day the detector is in a non-STABLE state, a planner must review the SKU.
We assign a **review cost per alert day** — the time and attention cost of
checking and deciding that no action is needed (a false positive) or acting on
a true positive.

### Missed-detection cost
If a real demand-regime break goes undetected, inventory may be mis-positioned
leading to stockout.  We assign a **stockout cost per undetected day** — each
day after the break where the detector is still in STABLE state represents
missed opportunity to adjust.

### Asymmetry
Stockout costs typically exceed overstock costs in supply chains where lost
sales are more harmful than holding cost.  We model this asymmetry explicitly.

> **These numbers are illustrative only.**  Real values depend on SKU margin,
> stockout policy, planner wage, and business context.  Do not use these
> numbers as production estimates.

In [ ]:
# Hypothetical cost parameters (USD per day, per SKU)
REVIEW_COST_PER_DAY   = 50    # cost of a planner reviewing one SKU in alert state
STOCKOUT_COST_PER_DAY = 300   # cost of a day of undetected regime change (missed detection)

# Note: stockout/review ratio = 6:1 here.
# At 1:1 (equal costs) or below, alert suppression is always better.
# At high ratios, missing a real break is very expensive.

print('Cost assumptions:')
print(f'  Review cost per alert day : ${REVIEW_COST_PER_DAY}')
print(f'  Stockout cost per missed day: ${STOCKOUT_COST_PER_DAY}')
print(f'  Stockout / review ratio     : {STOCKOUT_COST_PER_DAY / REVIEW_COST_PER_DAY:.1f}x')

## 2. Generate synthetic demand series

In [ ]:
rng = np.random.default_rng(42)
N         = 400
BREAK_DAY = 200

# Stable series (no break)
stable_series = rng.normal(loc=100.0, scale=10.0, size=N).clip(0)

# Mean-shift series (break at day 200)
pre  = rng.normal(loc=100.0, scale=10.0, size=BREAK_DAY).clip(0)
post = rng.normal(loc=160.0, scale=12.0, size=N - BREAK_DAY).clip(0)
shift_series = np.concatenate([pre, post])

print(f'Stable series  : mean={stable_series.mean():.1f}')
print(f'Shift series   : pre={shift_series[:BREAK_DAY].mean():.1f}  post={shift_series[BREAK_DAY:].mean():.1f}')

## 3. Run detectors

In [ ]:
det_base = DemandStateDetector()
det_v43  = StabilizedDemandDetectorV43()

# Stable series
df_base_stable = det_base.detect_rolling(stable_series)
df_v43_stable  = det_v43.detect_rolling_stabilized(stable_series)

# Mean-shift series
df_base_shift  = det_base.detect_rolling(shift_series)
df_v43_shift   = det_v43.detect_rolling_stabilized(shift_series)

print('Detection runs complete.')

## 4. Cost calculation

We compute two cost components per detector per scenario:

1. **Review cost** = (alert days) × REVIEW_COST_PER_DAY
2. **Missed-detection cost** = (days after break where detector is STABLE) × STOCKOUT_COST_PER_DAY

For the stable scenario there is no break, so the missed-detection cost is zero.
For the shift scenario, review days before the break are false positives, and
STABLE days after the break are misses.

In [ ]:
def compute_costs(df, break_day=None,
                  review_cost=REVIEW_COST_PER_DAY,
                  stockout_cost=STOCKOUT_COST_PER_DAY):
    """
    Compute illustrative cost metrics from a detection result DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Output of detect_rolling or detect_rolling_stabilized.
    break_day : int or None
        Day of injected break.  None for no-break (stable) scenario.
    review_cost : float
        Cost per day spent in a non-STABLE state.
    stockout_cost : float
        Cost per day after the break where the detector is still STABLE
        (missed detection).

    Returns
    -------
    dict
    """
    alert_days  = int((df['state'] != 'STABLE').sum())
    review_bill = alert_days * review_cost

    if break_day is None:
        missed_days    = 0
        stockout_bill  = 0.0
    else:
        post = df[df['t'] >= break_day]
        missed_days   = int((post['state'] == 'STABLE').sum())
        stockout_bill = missed_days * stockout_cost

    return {
        'alert_days':    alert_days,
        'review_cost':   review_bill,
        'missed_days':   missed_days,
        'stockout_cost': stockout_bill,
        'total_cost':    review_bill + stockout_bill,
    }


# --- Stable scenario costs ---
costs_base_stable = compute_costs(df_base_stable, break_day=None)
costs_v43_stable  = compute_costs(df_v43_stable,  break_day=None)

# --- Mean-shift scenario costs ---
costs_base_shift  = compute_costs(df_base_shift,  break_day=BREAK_DAY)
costs_v43_shift   = compute_costs(df_v43_shift,   break_day=BREAK_DAY)

cost_table = pd.DataFrame([
    {'Scenario': 'stable',     'Detector': 'Baseline', **costs_base_stable},
    {'Scenario': 'stable',     'Detector': 'V4.3',     **costs_v43_stable},
    {'Scenario': 'mean_shift', 'Detector': 'Baseline', **costs_base_shift},
    {'Scenario': 'mean_shift', 'Detector': 'V4.3',     **costs_v43_shift},
])

for col in ['review_cost', 'stockout_cost', 'total_cost']:
    cost_table[col] = cost_table[col].map('${:,.0f}'.format)

print('Illustrative cost comparison (single seed, synthetic data):')
print(f'  review_cost_per_day = ${REVIEW_COST_PER_DAY}')
print(f'  stockout_cost_per_day = ${STOCKOUT_COST_PER_DAY}')
print()
print(cost_table.to_string(index=False))

## 5. Sensitivity to cost ratio

The relative merit of a lower-alert detector versus a lower-delay detector
depends critically on the stockout-to-review cost ratio.  We sweep this ratio
across a range to show when V4.3 dominates (or does not).

In [ ]:
ratios = [0.5, 1, 2, 4, 6, 10, 20]
review_cost_fixed = 50   # keep review cost fixed, vary stockout

sensitivity_rows = []
for ratio in ratios:
    sc = ratio * review_cost_fixed  # stockout cost per day

    c_base = compute_costs(df_base_shift, BREAK_DAY,
                           review_cost=review_cost_fixed, stockout_cost=sc)
    c_v43  = compute_costs(df_v43_shift,  BREAK_DAY,
                           review_cost=review_cost_fixed, stockout_cost=sc)

    sensitivity_rows.append({
        'stockout/review ratio': ratio,
        'Baseline total cost ($)': c_base['total_cost'],
        'V4.3 total cost ($)': c_v43['total_cost'],
        'V4.3 saving ($)': c_base['total_cost'] - c_v43['total_cost'],
        'V4.3 is cheaper': c_v43['total_cost'] < c_base['total_cost'],
    })

sens_df = pd.DataFrame(sensitivity_rows)
for col in ['Baseline total cost ($)', 'V4.3 total cost ($)', 'V4.3 saving ($)']:
    sens_df[col] = sens_df[col].map('{:,.0f}'.format)

print('Sensitivity: V4.3 vs Baseline on mean-shift scenario')
print('(single seed, synthetic data, review_cost_per_day=$50)')
print()
print(sens_df.to_string(index=False))
print()
print('When V4.3 has fewer alert days but more missed days, the result flips')
print('depending on the stockout/review cost ratio.')

## 6. What the cost model shows (and does not show)

### What it shows

- **The economic tradeoff exists.**  A detector with lower false-alert burden
  (V4.3) reduces review cost on stable series and pre-break periods.
- **The tradeoff is assumption-sensitive.**  When stockout cost is high relative
  to review cost, slow detection (more missed days) can easily outweigh the
  savings from fewer false alerts.
- **V4.3 is not universally cheaper.**  For scenarios with slow break detection
  and high stockout cost, the Baseline or V4.1 may be cheaper despite more
  false alerts.

### What it does not show

- **Real cost validation.**  All costs are hypothetical.  Real SKU margins,
  planner costs, and stockout rates vary enormously across businesses.
- **Multi-SKU effects.**  A planner managing hundreds of SKUs faces batch
  effects that are not modelled here.
- **Inventory optimisation.**  This model does not simulate the downstream
  replenishment decisions that would actually incur or avoid stockout costs.
- **Real-world demand patterns.**  All series are synthetic; real demand may
  have seasonality, promotions, and other structure that changes detection
  performance.

### Conclusion

The cost simulation is directional only.  It provides a framework for thinking
about when alert-suppression improvements translate to economic value, and when
the detection delay they introduce is more costly than the false alerts they
prevent.  It should not be used to make product or procurement decisions.

In [ ]:
# Optional: plot cost sensitivity curve (requires matplotlib)
try:
    import matplotlib.pyplot as plt

    ratios_fine = np.linspace(0.2, 25, 100)
    review_cost_fixed = 50

    base_costs = []
    v43_costs  = []
    for ratio in ratios_fine:
        sc = ratio * review_cost_fixed
        base_costs.append(compute_costs(df_base_shift, BREAK_DAY,
                                        review_cost=review_cost_fixed, stockout_cost=sc)['total_cost'])
        v43_costs.append(compute_costs(df_v43_shift, BREAK_DAY,
                                       review_cost=review_cost_fixed, stockout_cost=sc)['total_cost'])

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ratios_fine, base_costs, label='Baseline', color='steelblue', lw=2)
    ax.plot(ratios_fine, v43_costs,  label='V4.3',     color='darkorange', lw=2)
    ax.fill_between(ratios_fine,
                    [b - v for b, v in zip(base_costs, v43_costs)],
                    0,
                    where=[b > v for b, v in zip(base_costs, v43_costs)],
                    alpha=0.15, color='green', label='V4.3 cheaper')
    ax.fill_between(ratios_fine,
                    [b - v for b, v in zip(base_costs, v43_costs)],
                    0,
                    where=[v > b for b, v in zip(base_costs, v43_costs)],
                    alpha=0.15, color='red', label='Baseline cheaper')
    ax.set_xlabel('Stockout cost / review cost ratio', fontsize=10)
    ax.set_ylabel('Total cost (USD)', fontsize=10)
    ax.set_title('Cost sensitivity: Baseline vs V4.3 on mean-shift scenario\n'
                 '(single seed, synthetic data, illustrative only)', fontsize=11)
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    plt.tight_layout()
    plt.savefig('cost_sensitivity.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Plot saved to cost_sensitivity.png')

except ImportError:
    print('matplotlib not installed – skipping sensitivity plot.')
    print('Run:  pip install matplotlib')

---

## Summary

This notebook documented a simple illustrative cost framing for the THUNBIT
detector:

- **Two cost types:** false-alert review cost and missed-detection stockout cost.
- **Key asymmetry:** stockout costs are typically higher than review costs, but
  the ratio is assumption-sensitive.
- **V4.3 is not always cheaper** than a faster, noisier detector; the answer
  depends on the cost ratio and the specific demand pattern.
- **Results are directional only** and should not be used for operational
  planning or procurement decisions.

For the full methodology and benchmark context see:
- `docs/methodology.md`
- `docs/benchmarking.md`
- `docs/limitations.md`